# Requirements Architect — dev smoke test

`mycontext.rac` turns a **natural-language product intent** into two linked,
eval-first specs and keeps them honest:

| Step | Function | What it produces |
|---|---|---|
| **Product** | `product(intent)` | the *what & why* — tasks, rubrics, action risk matrix, safety, gates |
| **Technical** | `technical(product=…)` | the *how* — architecture, guardrails, tools, cost, security… each `serves:` a product ID |
| **Trace** | `trace(product, technical)` | are the two specs in sync? (coverage + orphans) |

Sections 1–6 run **fully offline**. Section 7 shows `execute=True`, which uses
OpenAI (your own key) to **fill** the open questions into a complete spec.

In [1]:
# Dev setup -- make the in-repo `mycontext` importable without installing it.
import pathlib
import sys

try:
    import mycontext  # noqa: F401
except ModuleNotFoundError:
    here = pathlib.Path.cwd()
    for base in [here, *here.parents]:
        cand = base / "src"
        if (cand / "mycontext" / "__init__.py").exists():
            sys.path.insert(0, str(cand))
            break
    import mycontext  # noqa: F401

print("mycontext", mycontext.__version__, "->", pathlib.Path(mycontext.__file__).parent)

mycontext 0.12.0 -> C:\Users\dpokh\Desktop\mycontext\src\mycontext


## 1. The natural-language intent

A paragraph of product intent is all the architect needs to start.

In [3]:
INTENT = (
    "Brightcart is an online retailer. Support receives ~2,000 emails/day. "
    "Leadership wants Aurora: an agent that reads each email, looks up the "
    "customer's order, and drafts (eventually sends) a reply — with refunds "
    "requiring human approval. Success means faster replies and lower cost "
    "without wrong refunds, leaked customer data, or brand-damaging replies."
)
print(INTENT)

Brightcart is an online retailer. Support receives ~2,000 emails/day. Leadership wants Aurora: an agent that reads each email, looks up the customer's order, and drafts (eventually sends) a reply — with refunds requiring human approval. Success means faster replies and lower cost without wrong refunds, leaked customer data, or brand-damaging replies.


## 2. Product requirements (the *what & why*)

Gaps never block generation — anything that can't be inferred becomes an
`open_questions` entry + an inline `TODO(OQ-n)` marker.

In [4]:
from mycontext.rac import product, to_yaml, validate

prod = product(INTENT)
print("spec_type:", prod["meta"]["spec_type"], "| system:", prod["meta"]["system_name"])
print("tasks:   ", {k: v["name"] for k, v in prod["tasks"].items()})
print("actions: ", [(a["id"], a.get("policy")) for a in prod["actions"]])
print("safety:  ", [p["id"] for p in prod["safety"]])
print("\nvalidation:")
for i in validate(prod):
    print(" ", i)

spec_type: product_requirements | system: aurora
tasks:    {'T1': 'primary_request', 'T_oos': 'out_of_scope'}
actions:  [('A-1', 'auto'), ('A-2', 'auto'), ('A-3', 'approve'), ('A-4', 'approve'), ('A-X', 'forbidden')]
safety:   ['P1', 'P2', 'P3', 'P4']

validation:
  [ERROR] 7 blocking open question(s) unresolved: OQ-02, OQ-03, OQ-05, OQ-06, OQ-08, OQ-10, OQ-12.
  [WARN] 7 non-blocking open question(s) to review: OQ-04, OQ-07, OQ-09, OQ-11, OQ-13, OQ-14, OQ-15.


In [33]:
print(to_yaml(prod))

meta:
  system_name: aurora
  spec_type: product_requirements
  kind: agent
  spec_version: 0.1.0
  status: draft
  generated_by: mycontext-ai 0.12.0 requirements-architect
  tier: 1
  intent: 'Brightcart is an online retailer. Support receives ~2,000 emails/day. Leadership wants Aurora:
    an agent that reads each email, looks up the customer''s order, and drafts (eventually sends) a reply
    — with refunds requiring human approval. Success means faster replies and lower cost without wrong
    refunds, leaked customer data, or brand-damaging replies.'
  note: Authored by mycontext (authoring + scoring only). Enforcement — spec compilation, CI gates, human-in-the-loop
    approval, and budget/policy checks — belongs to your own stack / SDD tool (Spec Kit, Kiro, Claude
    Code, Cursor).
  review_checklist:
  - Replace every TODO(OQ-n) with a real value (see open_questions)
  - Confirm task-type frequencies from a real sample
  - Calibrate judge rubrics (>=80% agreement) before trusti

## 3. Technical requirements (the *how*)

Derived from the product spec, so every control `serves:` a concrete product ID
(`T*`, `A*`, `P*`, `G*`). `frontier=True` adds the fine-tune / RL / computer-use
layer.

In [34]:
from mycontext.rac import technical

tech = technical(product=prod, frontier=True)
print("spec_type:", tech["meta"]["spec_type"], "| source:", tech["meta"]["source"])
print("sections:", [k for k in tech if k not in ("meta", "assumptions", "open_questions")])
print("\narchitecture.pattern:", tech["architecture"]["pattern"])
print("output guardrails:")
for g in tech["guardrails"]["output"]:
    print(f"   - {g['control']}  -> serves {g['serves']}")

spec_type: technical_requirements | source: product_requirements
sections: ['architecture', 'guardrails', 'tools', 'cost', 'deployment', 'observability', 'security', 'failure_behavior', 'frontier']

architecture.pattern: tool_augmented_agent (plan -> act -> observe loop)
output guardrails:
   - Groundedness check — every claim traceable to source before send  -> serves ['R-T1']
   - No claim that an approval-gated action completed before approval  -> serves ['A-3', 'A-4']
   - PII / cross-customer data redaction on output  -> serves ['P2']
   - PII / cross-customer data redaction on output  -> serves ['P4']


In [35]:
print(to_yaml(tech))

meta:
  system_name: aurora
  spec_type: technical_requirements
  for_product: aurora
  kind: agent
  spec_version: 0.1.0
  status: draft
  generated_by: mycontext-ai 0.12.0 technical-architect
  source: product_requirements
  note: Technical requirements authored by mycontext. Each control's `serves` field links to the product
    requirement it implements. Authoring + scoring only — build/enforcement belongs to your stack.
  review_checklist:
  - Pin concrete model IDs, infra, and thresholds (replace TODO markers)
  - Confirm every `serves` reference resolves to a real product requirement
  - Run `mycontext rac trace` to verify product/technical coverage
architecture:
  pattern: tool_augmented_agent (plan -> act -> observe loop)
  orchestration:
    loop: plan -> act -> observe
    max_steps: TODO(OQ-01)
  state:
    short_term: per-request scratchpad (conversation memory)
    long_term: none (stateless between requests)
  model_routing_ref: cost.routing
  serves:
  - T1
guardrails:


## 4. Are the product & technical requirements in sync?

This is the core check — **no code involved**. `trace(product, technical)` confirms
every risk-bearing product requirement (tasks, safety, gated/forbidden actions,
gates) is covered by a technical control that `serves:` it, and flags any
technical control that references a product ID that doesn't exist (an orphan).

- `IN_SYNC` + `Coverage: N/N (100%)` → the two documents agree.
- Uncovered requirement → the product asked for something the technical spec
  doesn't address yet.
- Orphan → the technical spec built something no product requirement asked for.

In [36]:
from mycontext.rac import format_report, trace

sync = trace(prod, tech)   # <-- no diff: pure product <-> technical alignment
print(format_report(sync))

# Requirements trace — IN_SYNC

Coverage: 12/12 risk-bearing requirements served (100%).

## Findings

- [OK] Product and technical requirements are in sync.



### 4b. Optional: catch drift in a *code change*

`trace()` can *also* take a unified diff to check whether a code change still
respects the spec (e.g. introduces a forbidden action).

> ⚠️ The `DIFF` below is a **hand-written illustration** — `handlers/reply.py`
> is fictional, not a file in this repo. In real use you'd pipe in
> `git diff` output. Skip this cell if you only care about spec-to-spec sync.

In [37]:
EXAMPLE_DIFF = '''diff --git a/handlers/reply.py b/handlers/reply.py
--- a/handlers/reply.py
+++ b/handlers/reply.py
@@ -1,3 +1,6 @@
+def auto_handle(order):
+    issue_refund_or_payment(order)   # money action
+    delete_record(order.id)          # forbidden!
'''

drift = trace(prod, tech, diff=EXAMPLE_DIFF)
print(format_report(drift))

# Requirements trace — DRIFT_DETECTED

Coverage: 12/12 risk-bearing requirements served (100%).

## Findings

- [ERROR] handlers/reply.py: introduces forbidden tool/action 'delete_record' (policy=forbidden in product spec).

## Diff impact

- `handlers/reply.py` touches A-4, A-X — FORBIDDEN: delete_record



## 5. Projections — one spec, every coding agent

Render either spec into whatever a downstream tool expects. (`adr` is the
technical-architecture summary; `spec-kit` / `kiro` apply to the product spec.)

In [38]:
from mycontext.rac import project

print("========== AGENTS.md (from product) ==========")
print(project(prod, "agents-md"))
print("\n========== ADR / architecture.md (from technical) ==========")
print(project(tech, "adr"))

========== AGENTS.md (from product) ==========
# AGENTS.md — aurora

> Generated by mycontext requirements-architect from `requirements.yaml`.
> `requirements.yaml` is the source of truth; regenerate this file when it changes.

## Intent

Brightcart is an online retailer. Support receives ~2,000 emails/day. Leadership wants Aurora: an agent that reads each email, looks up the customer's order, and drafts (eventually sends) a reply — with refunds requiring human approval. Success means faster replies and lower cost without wrong refunds, leaked customer data, or brand-damaging replies.

## Operating rules (the constitution)

- NEVER call these tools (forbidden): delete_record, modify_history, merge_records.
- Require human approval before: send_reply.
- Require human approval before: issue_refund_or_payment.
- Safety: The system must never do: wrong refunds (defense in depth)
- Safety: The system must never do: leaked customer data (defense in depth)
- Safety: The system must never do: 

## 6. Bootstrap technical from intent alone (no product spec)

You can also generate technical requirements straight from natural language —
`serves` links are then left as `TODO(OQ-n)` nudging you to author a product spec
first for full traceability.

In [43]:
tech_nl = technical("A RAG assistant answers policy questions from our wiki; never invent answers.")
print("source:", tech_nl["meta"]["source"], "| pattern:", tech_nl["architecture"]["pattern"])

source: natural_language | pattern: retrieve_then_generate (RAG over a trusted source)


## 7. Execute with OpenAI — the **complete** requirements

`execute=True` sends each spec's `open_questions` to OpenAI (via LiteLLM, your own
key) and substitutes the answers back in — so you get a *filled* requirements doc
instead of a skeleton with `TODO(OQ-n)` markers. It's also **pattern-driven**: a
curated set of cognitive patterns (task decomposition, rubric design, pre-mortem
for `product`; architecture/trade-off, risk, ethics for `technical`) runs over the
intent first, and those analyses are fed to the fill model as expert grounding —
the raw analyses are attached under `_pattern_notes`. The flow is:

1. **product** with `execute=True` → filled product requirements
2. **technical** from that filled product, also `execute=True` → filled technical requirements
3. **trace** the two filled specs to confirm they're still in sync

Set `OPENAI_API_KEY` in your environment first. If it's not set, the cells fall
back to the offline drafts so the notebook still runs end-to-end.

In [44]:
import os

PROVIDER = "openai"
MODEL = "gpt-4o-mini"  # cheap + capable; any LiteLLM-supported model works

# Provide your key via the environment — never hard-code it in the notebook:
#   PowerShell:  $env:OPENAI_API_KEY = "sk-..."
#   bash/zsh:    export OPENAI_API_KEY="sk-..."
assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your environment before running section 7."


### 7a. Generate the product requirements (filled)

In [ ]:
prod_full = product(INTENT, execute=True, provider=PROVIDER, model=MODEL)
answered = [q for q in prod_full["open_questions"] if q.get("status") == "answered"]
print("status:", prod_full["meta"]["status"], "| answered", len(answered), "open questions")
print(to_yaml(prod_full))

### 7a-bis. What ran under the hood — the cognitive-pattern brief

`execute=True` doesn't just call one fill model. First it runs a curated set of
cognitive patterns over the intent (for `product`: **problem_decomposer** +
**gap_analyzer** → `tasks`, **rubric_designer** → `rubrics`,
**scenario_planner** + **root_cause_analyzer** → `safety`). Their analyses are
fed to the fill model as expert grounding and then *distilled* into the answers.

The spec stays clean — it records only a lightweight `meta.informed_by`
provenance map. To read the analyses themselves, use the public helpers (also on
the CLI as `mycontext rac analyze "<intent>"`):

In [ ]:
from IPython.display import Markdown, display

from mycontext.rac import analyze, format_brief

# The clean provenance the spec keeps:
print("meta.informed_by:", prod_full["meta"].get("informed_by"))

# The analyses themselves, rendered as readable markdown (not dumped into the spec):
brief = analyze(INTENT, kind="product", provider=PROVIDER, model=MODEL)
display(Markdown(format_brief(brief)))

### 7b. Generate the technical requirements from the filled product (filled)

In [ ]:
tech_full = technical(product=prod_full, frontier=True, execute=True, provider=PROVIDER, model=MODEL)
print("status:", tech_full["meta"]["status"], "| filled_by:", tech_full["meta"].get("filled_by"))
print("technical sections informed by patterns:", tech_full["meta"].get("informed_by"))
print(to_yaml(tech_full))

### 7c. Confirm the two filled specs are still in sync

In [ ]:
print(format_report(trace(prod_full, tech_full)))